# 面试问题：Constitutional AI / RLAIF 怎样把原则变成可训练、可审计的数据？

**回答主线。** Constitution 不是一段抽象口号，而是有 ID、优先级、适用范围、判定标准和升级路径的版本化策略。典型流程先让模型依据原则自我 critique 与 revision，形成监督微调数据；再让 AI judge 比较候选，生成偏好对，训练 preference/reward model，最后进入 RL 或直接偏好优化。

下面用确定性小规则模拟数据管线，重点验证 provenance、冲突、位置偏差、reward 拟合和发布门禁。规则模拟器不代表真实安全分类器，也不会把 AI feedback 当成人类监督的无条件替代品。


In [ ]:
import hashlib, json, math, re
from dataclasses import dataclass
import numpy as np

# 固定随机源只用于数值优化和抽样审计。
rng157 = np.random.default_rng(157)

def canonical_digest157(value):
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()

assert len(canonical_digest157({"a": 1})) == 64
assert canonical_digest157({"a": 1, "b": 2}) == canonical_digest157({"b": 2, "a": 1})
assert canonical_digest157({"a": 1}) != canonical_digest157({"a": 2})


## 1. Principle 需要稳定 ID、优先级与可执行判定

自然语言原则允许解释，但生产管线仍要给出机器可检查的触发器、动作和 human escalation。相同 ID 不可覆盖；优先级相同且动作冲突时不能静默取输入顺序。


In [ ]:
@dataclass(frozen=True)
class Principle157:
    principle_id: str
    priority: int
    pattern: str
    action: str
    scope: str = "all"

def validate_constitution157(principles):
    # ID 是日志和训练样本的外键，必须唯一且字段完整。
    ids = [p.principle_id for p in principles]
    if len(ids) != len(set(ids)):
        raise ValueError("duplicate principle id")
    if any(p.priority < 0 or p.action not in {"redact", "refuse", "revise", "escalate"} for p in principles):
        raise ValueError("invalid principle")
    return sorted(principles, key=lambda p: (-p.priority, p.principle_id))

constitution157 = validate_constitution157([
    Principle157("P-SECRET", 100, r"sk-[A-Za-z0-9]+", "redact"),
    Principle157("P-ABUSE", 50, r"\bidiot\b", "revise"),
    Principle157("P-WEAPON", 90, r"build a weapon", "refuse"),
])
assert [p.principle_id for p in constitution157] == ["P-SECRET", "P-WEAPON", "P-ABUSE"]
assert constitution157[0].priority == 100
assert len({p.principle_id for p in constitution157}) == 3


## 2. Critique 输出证据 span，而不是只给笼统分数

每个 violation 记录 principle、匹配文本和位置，方便复核与重放。Judge 的解释不能反过来成为事实源；真正的判定仍绑定原始 response 和 constitution revision。


In [ ]:
def critique157(response, principles):
    # 使用受控 regex 返回可定位证据；真实系统会组合分类器与人工审核。
    findings = []
    for principle in principles:
        for match in re.finditer(principle.pattern, response, flags=re.I):
            findings.append({"principle_id": principle.principle_id, "action": principle.action, "span": match.span(), "evidence": match.group(0)})
    return findings

unsafe157 = "You are an idiot. Use sk-ABC123 and build a weapon."
findings157 = critique157(unsafe157, constitution157)
assert {f["principle_id"] for f in findings157} == {"P-SECRET", "P-ABUSE", "P-WEAPON"}
assert all(f["span"][0] < f["span"][1] for f in findings157)
assert any(f["evidence"] == "sk-ABC123" for f in findings157)


## 3. Revision 应最小修改并保留有帮助的安全内容

简单整段拒答可能降低风险，却造成过度拒答。处理顺序按优先级：敏感值先脱敏，侮辱措辞改写；若命中高风险行为则拒绝具体步骤，并提供安全替代方向。


In [ ]:
def revise157(response, principles):
    # 每条原则只执行声明动作，避免 revision 模型自行扩大政策。
    revised = response
    applied = []
    for principle in principles:
        if not re.search(principle.pattern, revised, flags=re.I):
            continue
        applied.append(principle.principle_id)
        if principle.action == "redact":
            revised = re.sub(principle.pattern, "[REDACTED]", revised, flags=re.I)
        elif principle.action == "revise":
            revised = re.sub(principle.pattern, "unhelpful", revised, flags=re.I)
        elif principle.action == "refuse":
            revised = re.sub(principle.pattern, "discuss safe prevention", revised, flags=re.I)
    return revised, applied

revised157, applied157 = revise157(unsafe157, constitution157)
assert "sk-" not in revised157 and "idiot" not in revised157.lower()
assert "safe prevention" in revised157
assert set(applied157) == {"P-SECRET", "P-ABUSE", "P-WEAPON"}


## 4. 偏好对绑定 prompt、候选、原则和生成版本

只保存 chosen/rejected 文本无法解释选择，也无法在 constitution 更新后重算。样本摘要应覆盖 prompt、两候选、适用 principle、生成模型和 judge revision，并按 prompt family 切分避免泄漏。


In [ ]:
@dataclass(frozen=True)
class PreferencePair157:
    prompt_id: str
    chosen: str
    rejected: str
    principle_ids: tuple
    generator_revision: str
    judge_revision: str
    digest: str

def make_pair157(prompt_id, chosen, rejected, principle_ids, generator, judge):
    # digest 覆盖语义字段，后续任何文本或版本变化都会产生新样本。
    record = {"prompt_id": prompt_id, "chosen": chosen, "rejected": rejected, "principles": sorted(principle_ids), "generator": generator, "judge": judge}
    return PreferencePair157(prompt_id, chosen, rejected, tuple(sorted(principle_ids)), generator, judge, canonical_digest157(record))

pair157 = make_pair157("prompt-family-7", revised157, unsafe157, applied157, "policy-v2", "judge-v5")
assert pair157.chosen != pair157.rejected
assert len(pair157.digest) == 64
assert pair157.principle_ids == tuple(sorted(applied157))


## 5. AI Judge 必须做位置交换、tie 与多评审一致性

同一对候选按 A/B 与 B/A 两个顺序评审；结论不一致时标记 tie/uncertain，而不是强造标签。多 judge 的版本和独立性同样重要，共享同一盲点的多数票没有可靠性增益。


In [ ]:
def safety_score157(response, principles):
    # 受控分数以 violation 数为主，并给安全替代内容少量 helpfulness 奖励。
    violations = len(critique157(response, principles))
    helpful = int("safe" in response.lower() or "prevention" in response.lower())
    return -2.0 * violations + 0.25 * helpful

def pair_judgment157(a, b, position_bias=0.0):
    score_a = safety_score157(a, constitution157) + position_bias
    score_b = safety_score157(b, constitution157)
    return "A" if score_a > score_b else "B" if score_b > score_a else "tie"

forward157 = pair_judgment157(revised157, unsafe157)
reverse157 = pair_judgment157(unsafe157, revised157)
assert forward157 == "A"
assert reverse157 == "B"
assert pair_judgment157("safe answer", "safe answer") == "tie"


## 6. Bradley–Terry 把 chosen/rejected 差分映射成偏好概率

Preference model 可学习 `sigmoid(r_chosen-r_rejected)`。下面用三维可解释特征和梯度下降验证 loss 下降；真实 reward model 还要做 prompt split、tie、校准和分布外测试。


In [ ]:
def feature157(text):
    # 特征依次为负 violation、是否含安全替代、截断长度。
    return np.array([-len(critique157(text, constitution157)), int("safe" in text.lower()), min(len(text), 200) / 200.0], dtype=np.float64)

pairs157 = [(revised157, unsafe157), ("safe prevention guidance", "build a weapon"), ("polite safe answer", "you idiot")]
differences157 = np.stack([feature157(c) - feature157(r) for c, r in pairs157])
weights157 = np.zeros(3)
def bt_loss157(w):
    margin = differences157 @ w
    return float(np.mean(np.logaddexp(0.0, -margin)))
initial_loss157 = bt_loss157(weights157)
for _ in range(200):
    margin = differences157 @ weights157
    gradient = -(differences157 * (1.0 / (1.0 + np.exp(margin)))[:, None]).mean(axis=0)
    weights157 -= 0.1 * gradient
assert bt_loss157(weights157) < initial_loss157
assert np.all(differences157 @ weights157 > 0)
assert np.isfinite(weights157).all()


## 7. 原则冲突与低置信必须升级给人

两条同优先级原则若对同一 span 要求不同动作，系统没有足够信息自行裁决。高影响领域还应配置强制人工 slice，即使 judge 分数很高也不能自动发布。


In [ ]:
def resolve_actions157(findings, principles):
    # 同一证据按最高优先级裁决；最高层动作不唯一则显式 escalate。
    priority = {p.principle_id: p.priority for p in principles}
    grouped = {}
    for finding in findings:
        grouped.setdefault(finding["span"], []).append(finding)
    decisions = []
    for span, group in grouped.items():
        top = max(priority[f["principle_id"]] for f in group)
        actions = {f["action"] for f in group if priority[f["principle_id"]] == top}
        decisions.append("escalate" if len(actions) > 1 else next(iter(actions)))
    return decisions

conflict_principles157 = [Principle157("A", 10, "x", "redact"), Principle157("B", 10, "x", "refuse")]
conflict_findings157 = critique157("x", conflict_principles157)
assert resolve_actions157(conflict_findings157, conflict_principles157) == ["escalate"]
assert resolve_actions157(findings157, constitution157).count("escalate") == 0
assert len(resolve_actions157(findings157, constitution157)) == 3


## 8. 发布门禁同时看风险下降、帮助性与过度拒答

只优化 harmlessness 会得到“什么都不回答”的退化模型。回归集要包含应拒绝、应改写和完全安全三类，并按语言、领域和脆弱群体切片；AI judge 指标必须用独立人审校准。


In [ ]:
def release_gate157(before, after, safe_prompts_retained, overrefusal_rate):
    # 风险下降是必要条件，同时限制安全请求的帮助性损失和过度拒答。
    before_v = sum(len(critique157(x, constitution157)) for x in before)
    after_v = sum(len(critique157(x, constitution157)) for x in after)
    metrics = {"violation_reduction": (before_v - after_v) / max(before_v, 1), "safe_retention": safe_prompts_retained, "overrefusal": overrefusal_rate}
    passed = metrics["violation_reduction"] >= 0.8 and metrics["safe_retention"] >= 0.95 and metrics["overrefusal"] <= 0.05
    return passed, metrics

passed157, metrics157 = release_gate157([unsafe157], [revised157], 0.98, 0.03)
assert passed157
assert metrics157["violation_reduction"] == 1.0
assert not release_gate157([unsafe157], [revised157], 0.8, 0.2)[0]


## 面试总结

- Constitution 要有稳定 ID、优先级、scope、动作、冲突和人审升级，而不是只写价值观口号。
- Critique/revision 生成 SFT 数据；AI comparison 生成偏好数据，二者都要绑定原响应、原则和模型版本。
- Judge 要检查位置偏差、tie、独立性和人类校准；AI feedback 不能无条件升级成真值。
- 发布门禁同时约束 violation、helpfulness、over-refusal、slice 与人工审核。

延伸阅读：[Constitutional AI](https://arxiv.org/abs/2212.08073)、[Scaling Laws for RLAIF](https://arxiv.org/abs/2309.00267)、[Training Language Models to Follow Instructions with Human Feedback](https://arxiv.org/abs/2203.02155)。
